In [1]:
# Comparación de Modelos Predictivos y Entrenamiento de modelos
## Random Forest, Gradient Boosting, XGBoost

In [10]:
import sys
!{sys.executable} -m pip install optuna
!{sys.executable} -m pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [9]:
import sys
import os
# Agregar la raíz del proyecto al PATH
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
import warnings

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import TimeSeriesSplit
from src.preprocessing import preparar_dataset, FEATURES
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

In [10]:
#Cargar dataset
df, X, y = preparar_dataset(
    "../data/raw/dataset_sintetico_demanda_lima.xlsx"
)

In [11]:
# Dividir datos (80% entrenamiento, 20% prueba)

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]
# Validación cruzada temporal
tscv = TimeSeriesSplit(n_splits=5)
print(f"Datos: {len(df)} días")
print(f"Train: {len(X_train)} días, Test: {len(X_test)} días")
print(f"Features: {len(FEATURES)}")


Datos: 709 días
Train: 567 días, Test: 142 días
Features: 21


In [12]:
# ============================================
# FUNCIÓN PARA CALCULAR LAS 5 MÉTRICAS
# ============================================
def calcular_metricas(y_real, y_pred, n_features):
    mae = mean_absolute_error(
        y_real,
        y_pred
    )
    rmse = np.sqrt(
        mean_squared_error(
            y_real,
            y_pred
        )
    )
    mape = (
        np.mean(
            np.abs(
                (y_real - y_pred)
                / y_real
            )
        ) * 100
    )
    r2 = r2_score(
        y_real,
        y_pred
    )
    n = len(y_real)
    r2_adj = (
        1 -
        (
            (1-r2)*(n-1)
            /
            (n-n_features-1)
        )
    )
    return {
        "MAE": round(mae,2),
        "RMSE": round(rmse,2),
        "MAPE": round(mape,2),
        "R2": round(r2,4),
        "R2_Ajustado": round(r2_adj,4)
    }

In [13]:
#OPTUNA PARA RANDOM FOREST
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            500
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            20
        ),
        "min_samples_split": trial.suggest_int(
            "min_samples_split",
            2,
            10
        )
    }

    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    mae_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        pred = model.predict(X_val)

        mae_scores.append(
            mean_absolute_error(y_val, pred)
        )

    return np.mean(mae_scores)

In [14]:
#Ejecución para Random Forest
study_rf = optuna.create_study(
    direction="minimize"
)
study_rf.optimize(
    objective_rf,
    n_trials=30
)
study_rf.best_params

[I 2026-07-01 16:16:08,782] A new study created in memory with name: no-name-b34c2c1e-93bf-401c-a98c-c54cd8a29b16
[I 2026-07-01 16:16:10,783] Trial 0 finished with value: 9.668659673300088 and parameters: {'n_estimators': 269, 'max_depth': 17, 'min_samples_split': 6}. Best is trial 0 with value: 9.668659673300088.
[I 2026-07-01 16:16:12,277] Trial 1 finished with value: 11.754926838716376 and parameters: {'n_estimators': 210, 'max_depth': 4, 'min_samples_split': 10}. Best is trial 0 with value: 9.668659673300088.
[I 2026-07-01 16:16:15,354] Trial 2 finished with value: 9.503894708373046 and parameters: {'n_estimators': 424, 'max_depth': 9, 'min_samples_split': 4}. Best is trial 2 with value: 9.503894708373046.
[I 2026-07-01 16:16:18,552] Trial 3 finished with value: 9.767402691043845 and parameters: {'n_estimators': 436, 'max_depth': 11, 'min_samples_split': 8}. Best is trial 2 with value: 9.503894708373046.
[I 2026-07-01 16:16:21,722] Trial 4 finished with value: 9.518963709964023 and

{'n_estimators': 496, 'max_depth': 20, 'min_samples_split': 2}

In [15]:
#OPTUNA PARA GRADIENT BOOSTING
def objective_gb(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            500
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            8
        )
    }

    model = GradientBoostingRegressor(
        **params,
        random_state=42
    )

    mae_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        pred = model.predict(X_val)

        mae_scores.append(
            mean_absolute_error(y_val, pred)
        )

    return np.mean(mae_scores)

In [16]:
#Ejecución para Gradient Boosting
study_gb = optuna.create_study(
    direction="minimize"
)
study_gb.optimize(
    objective_gb,
    n_trials=30
)
study_gb.best_params

[I 2026-07-01 16:18:07,542] A new study created in memory with name: no-name-d838492f-8d43-44b6-bdbb-d6a1a42436a9
[I 2026-07-01 16:18:08,822] Trial 0 finished with value: 6.857909048755492 and parameters: {'n_estimators': 150, 'learning_rate': 0.2643149219849986, 'max_depth': 3}. Best is trial 0 with value: 6.857909048755492.
[I 2026-07-01 16:18:13,500] Trial 1 finished with value: 7.249010504780711 and parameters: {'n_estimators': 476, 'learning_rate': 0.1108891783590189, 'max_depth': 4}. Best is trial 0 with value: 6.857909048755492.
[I 2026-07-01 16:18:16,800] Trial 2 finished with value: 9.346137245212876 and parameters: {'n_estimators': 193, 'learning_rate': 0.03447187577098019, 'max_depth': 7}. Best is trial 0 with value: 6.857909048755492.
[I 2026-07-01 16:18:19,674] Trial 3 finished with value: 8.5477200088764 and parameters: {'n_estimators': 236, 'learning_rate': 0.27561260876801075, 'max_depth': 6}. Best is trial 0 with value: 6.857909048755492.
[I 2026-07-01 16:18:22,930] Tr

{'n_estimators': 288, 'learning_rate': 0.13025800251478986, 'max_depth': 2}

In [17]:
#OPTUNA PARA XGBOOST
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            500
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),
        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        )
    }

    model = XGBRegressor(
        **params,
        random_state=42,
        verbosity=0
    )

    mae_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        pred = model.predict(X_val)

        mae_scores.append(
            mean_absolute_error(y_val, pred)
        )

    return np.mean(mae_scores)

In [18]:
#Ejecución para XGBoost
study_xgb = optuna.create_study(
    direction="minimize"
)
study_xgb.optimize(
    objective_xgb,
    n_trials=30
)
study_xgb.best_params

[I 2026-07-01 16:19:29,293] A new study created in memory with name: no-name-d18637a1-fae3-41f3-a8d7-8e7fd856b22a
[I 2026-07-01 16:19:30,480] Trial 0 finished with value: 8.285064125061036 and parameters: {'n_estimators': 251, 'learning_rate': 0.09527483715448412, 'max_depth': 7, 'subsample': 0.9513672535128914}. Best is trial 0 with value: 8.285064125061036.
[I 2026-07-01 16:19:31,359] Trial 1 finished with value: 7.971847057342529 and parameters: {'n_estimators': 175, 'learning_rate': 0.12279524464724015, 'max_depth': 8, 'subsample': 0.8301825286608117}. Best is trial 1 with value: 7.971847057342529.
[I 2026-07-01 16:19:32,228] Trial 2 finished with value: 8.297413158416749 and parameters: {'n_estimators': 463, 'learning_rate': 0.29843525016131733, 'max_depth': 9, 'subsample': 0.8037652064161542}. Best is trial 1 with value: 7.971847057342529.
[I 2026-07-01 16:19:33,062] Trial 3 finished with value: 8.331807136535645 and parameters: {'n_estimators': 474, 'learning_rate': 0.2436817806

{'n_estimators': 199,
 'learning_rate': 0.13573449193198325,
 'max_depth': 3,
 'subsample': 0.6588121195103418}

In [19]:
#ENTREAMIENTO FINAL PARA LOS MODELOS
#1. Random Forest
rf = RandomForestRegressor(
    **study_rf.best_params,
    random_state=42
)
#2. Gradient Boosting
gb = GradientBoostingRegressor(
    **study_gb.best_params,
    random_state=42
)
#3. XGBoost
xgb = XGBRegressor(
    **study_xgb.best_params,
    random_state=42,
    verbosity=0
)

In [20]:
#Ejecución de entrenamiento de los modelos
rf.fit(X_train,y_train)
gb.fit(X_train,y_train)
xgb.fit(X_train,y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [21]:
#Predicciones
y_pred_rf = rf.predict(X_test)
y_pred_gb = gb.predict(X_test)
y_pred_xgb = xgb.predict(X_test)

In [22]:
#Métricas
metrics_rf = calcular_metricas(
    y_test,
    y_pred_rf,
    len(feature_cols)
)
metrics_gb = calcular_metricas(
    y_test,
    y_pred_gb,
    len(feature_cols)
)
metrics_xgb = calcular_metricas(
    y_test,
    y_pred_xgb,
    len(feature_cols)
)

In [23]:
#TABLA FINAL
resultados = pd.DataFrame([
    {
        "Modelo":"Random Forest",
        **metrics_rf
    },
    {
        "Modelo":"Gradient Boosting",
        **metrics_gb
    },
    {
        "Modelo":"XGBoost",
        **metrics_xgb
    }
])
resultados = resultados.sort_values(
    "MAE"
)
resultados

,Modelo,MAE,RMSE,MAPE,R2,R2_Ajustado
2,XGBoost,4.52,5.98,2.22,0.9675,0.9647
1,Gradient Boosting,4.68,6.08,2.30,0.9664,0.9635
0,Random Forest,7.16,10.25,3.50,0.9044,0.8963


In [24]:
#Guardar CSV
import os
os.makedirs(
    "../reports/tables",
    exist_ok=True
)
resultados.to_csv(
    "../reports/tables/model_comparison_full.csv",
    index=False
)

In [25]:
#Guardar modelos
os.makedirs(
    "../data/processed",
    exist_ok=True
)
joblib.dump(
    rf,
    "../data/processed/random_forest.pkl"
)
joblib.dump(
    gb,
    "../data/processed/gradient_boosting.pkl"
)
joblib.dump(
    xgb,
    "../data/processed/xgboost.pkl"
)

['../data/processed/xgboost.pkl']